# Molecular Melting Point Prediction using RDKit and Machine Learning

### About this Project

This project is my attempt to explore how **Machine Learning** can be used to predict the **melting point of molecules** from their chemical structure.

As someone interested in both **Chemistry** and **Machine Learning**, I wanted to work on a project that combines these two fields. Instead of using image or text datasets, this project focuses on molecular data represented as **SMILES** (Simplified Molecular Input Line Entry System).

Using the **RDKit** library, I will convert these molecular structures into numerical features (called molecular descriptors and fingerprints) that can be understood by Machine Learning models. I will then train different regression models and compare their performance in predicting melting points.

---

## What I Hope to Learn

Through this project, I aim to:

- Understand how molecular datasets are structured.
- Learn how RDKit represents molecules.
- Generate molecular descriptors and fingerprints.
- Build and evaluate Machine Learning regression models.
- Explore the connection between chemistry and data science.

---

## Project Workflow

1. Load and understand the dataset.
2. Clean and inspect the data.
3. Process SMILES using RDKit.
4. Generate molecular descriptors.
5. Train Machine Learning models.
6. Evaluate model performance.
7. Compare different approaches and summarize the results.

---

## Dataset

This project uses the **Jean-Claude Bradley Open Melting Point Dataset**, which contains experimentally measured melting points and molecular SMILES.


In [21]:
# Import necessary libraries

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Loading the Dataset
The objective is to:

- Load the Excel dataset.
- Check the dimensions of the dataset.
- View the first few rows.
- Understand the available columns before beginning data preprocessing.

In [22]:
dataset = pd.read_excel("BradleyMeltingPointDataset.xlsx")
dataset.head()

,key,name,smiles,mpC,csid,link,source,donotuse,donotusebecause
0,1,"2-(2,4-dinitrobenzyl)pyridine",c1ccnc(c1)Cc2ccc(cc2[N+](=O)[O-])[N+](=O)[O-],92.0,64018,http://www.alfa.com/en/GP100W.pgm?DSSTK=B24192,Alfa Aesar,NaN,NaN
1,2,2-(1-piperidinyl)aniline,c1ccc(c(c1)N)N2CCCCC2,46.0,403764,http://www.alfa.com/en/GP100W.pgm?DSSTK=A13073,Alfa Aesar,NaN,NaN
2,3,2-(1-piperazinyl)pyrimidine,c1cnc(nc1)N2CCNCC2,33.0,80080,http://www.alfa.com/en/GP100W.pgm?DSSTK=L15884,Alfa Aesar,NaN,NaN
3,4,2-(1-piperazinyl)phenol,c1ccc(c(c1)N2CCNCC2)O,125.0,63701,http://www.alfa.com/en/GP100W.pgm?DSSTK=B20252,Alfa Aesar,NaN,NaN
4,5,2-(1-cyclohexenyl)ethylamine,C1CCC(=CC1)CCN,-55.0,69388,http://www.alfa.com/en/GP100W.pgm?DSSTK=L08261,Alfa Aesar,NaN,NaN


In [23]:
dataset.shape

(28645, 9)

# 2. Data Cleaning (PRE OF Preprocessing)

Before training any Machine Learning model, it is important to clean the dataset. Real-world datasets often contain missing values, duplicate entries, invalid records, or information that is not useful for prediction.

In this phase, I will inspect the dataset and remove unnecessary or unreliable data while keeping the important information required for the project.

For this project, the two most important columns are:

- **name** – The common name of the molecule. This helps identify molecules while exploring the dataset.
- **smiles** – The molecular structure represented in SMILES format. This will be processed using RDKit to generate molecular features.
- **mpC** – The experimental melting point (in °C), which is the target variable for prediction.

The remaining columns contain identifiers, links, metadata, and quality information that are not required for model training. After cleaning, the dataset will be ready for molecular preprocessing and feature generation.

In [24]:
data = dataset[['name','smiles','mpC']]
data.head()

,name,smiles,mpC
0,"2-(2,4-dinitrobenzyl)pyridine",c1ccnc(c1)Cc2ccc(cc2[N+](=O)[O-])[N+](=O)[O-],92.0
1,2-(1-piperidinyl)aniline,c1ccc(c(c1)N)N2CCCCC2,46.0
2,2-(1-piperazinyl)pyrimidine,c1cnc(nc1)N2CCNCC2,33.0
3,2-(1-piperazinyl)phenol,c1ccc(c(c1)N2CCNCC2)O,125.0
4,2-(1-cyclohexenyl)ethylamine,C1CCC(=CC1)CCN,-55.0


In [25]:
# lets see if we have a new null values or not
data.isnull().sum()

name      0
smiles    0
mpC       0
dtype: int64

looks good if we don't have null values in data we have

# 3. Processing SMILES with RDKit

Machine Learning models cannot directly understand chemical structures written as **SMILES (Simplified Molecular Input Line Entry System)**. Therefore, the first step is to convert these text representations into molecular objects using the **RDKit** library.

In this phase, I will:

- Convert each SMILES string into an RDKit molecule (`Mol`) object.
- Check for invalid or corrupted SMILES strings.
- Remove molecules that cannot be processed correctly.
- Prepare the dataset for molecular descriptor and fingerprint generation in the next phase.

By the end of this phase, every row in the dataset will contain a valid molecular structure that can be analyzed computationally.

In [44]:
import rdkit.Chem as Chem

In [45]:
data['mol'] = data['smiles'].apply(Chem.MolFromSmiles)

as we observe in data we have certain abnormal points

In [28]:
# Count invalid molecules

invalid = data["mol"].isna().sum()

print(f"Invalid molecules: {invalid}")
print(f"Valid molecules  : {len(data) - invalid}")

Invalid molecules: 301
Valid molecules  : 28344


301 from 28645 its around 1.05% of data better messing up with this we should remove that fraction having data size of 28344 entries in it... still works good

In [29]:
data = data[data['mol'].notna()].reset_index(drop = True)

In [37]:
data.head()

,name,smiles,mpC,mol
0,"2-(2,4-dinitrobenzyl)pyridine",c1ccnc(c1)Cc2ccc(cc2[N+](=O)[O-])[N+](=O)[O-],92.0,<rdkit.Chem.rdchem.Mol object at 0x76798e0d5b60>
1,2-(1-piperidinyl)aniline,c1ccc(c(c1)N)N2CCCCC2,46.0,<rdkit.Chem.rdchem.Mol object at 0x76798e0d5bd0>
2,2-(1-piperazinyl)pyrimidine,c1cnc(nc1)N2CCNCC2,33.0,<rdkit.Chem.rdchem.Mol object at 0x76798e0d5c40>
3,2-(1-piperazinyl)phenol,c1ccc(c(c1)N2CCNCC2)O,125.0,<rdkit.Chem.rdchem.Mol object at 0x76798e0d5cb0>
4,2-(1-cyclohexenyl)ethylamine,C1CCC(=CC1)CCN,-55.0,<rdkit.Chem.rdchem.Mol object at 0x76798e0d5d20>


In [31]:
data.shape

(28344, 4)

# 4. Molecular Descriptor Generation

Machine Learning models cannot learn directly from molecular structures. They require numerical features as input.

In this phase, I will use **RDKit** to calculate molecular descriptors for each molecule. These descriptors describe different physical, chemical, and structural properties of a molecule, such as its molecular weight, number of atoms, hydrogen bond donors and acceptors, topological properties, and many others.

These numerical values will become the **features (X)** that the Machine Learning models use to predict the **melting point (mpC)**.

The objective of this phase is to convert every valid molecule into a set of meaningful numerical descriptors while preserving the melting point as the target variable.

In [49]:
from rdkit.Chem import Descriptors

descriptor_names = [name for name, _ in Descriptors.descList]

def calculate_all_descriptors(mol):
    return [func(mol) for _, func in Descriptors.descList]

descriptor_data = pd.DataFrame(
    data["mol"].apply(calculate_all_descriptors).tolist(),
    columns=descriptor_names
)

final_data = pd.concat(
    [data[["name", "smiles", "mpC", "mol"]], descriptor_data],
    axis=1
)

In [50]:
descriptor_data = descriptor_data.dropna(axis=1)

In [51]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0)

selector.fit(descriptor_data)

descriptor_data = descriptor_data.loc[:, selector.get_support()]

In [52]:
correlation = descriptor_data.corrwith(final_data["mpC"])

In [53]:
top_descriptors = correlation.head(30).index.tolist()

print(top_descriptors)

['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex', 'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'NumRadicalElectrons', 'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3', 'AvgIpc', 'BalabanJ', 'BertzCT', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3n', 'Chi3v', 'Chi4n', 'Chi4v', 'HallKierAlpha']


In [54]:
data = pd.concat(
    [
        final_data[["name", "smiles", "mpC"]],
        descriptor_data[top_descriptors]
    ],
    axis=1
)

In [55]:
data.head()

,name,smiles,mpC,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,NumRadicalElectrons,FpDensityMorgan1,FpDensityMorgan2,FpDensityMorgan3,AvgIpc,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,Chi1v,Chi2n,Chi2v,Chi3n,Chi3v,Chi4n,Chi4v,HallKierAlpha
0,"2-(2,4-dinitrobenzyl)pyridine",c1ccnc(c1)Cc2ccc(cc2[N+](=O)[O-])[N+](=O)[O-],92.0,10.949337,10.949337,0.259075,-0.655211,0.619888,10.105263,259.221,250.149,259.059306,96,0,1.052632,1.842105,2.421053,2.394721,2.399145,628.126552,13.828063,9.723193,9.723193,9.075387,5.393132,5.393132,3.825087,3.825087,2.574114,2.574114,1.683069,1.683069,-2.83
1,2-(1-piperidinyl)aniline,c1ccc(c(c1)N)N2CCCCC2,46.0,5.909724,5.909724,0.906852,0.906852,0.664933,17.384615,176.263,160.135,176.131349,70,0,1.000000,1.692308,2.384615,2.198429,2.183880,277.173697,9.096012,7.869499,7.869499,6.377010,4.972088,4.972088,3.606524,3.606524,2.633505,2.633505,1.910324,1.910324,-1.18
2,2-(1-piperazinyl)pyrimidine,c1cnc(nc1)N2CCNCC2,33.0,4.189444,4.189444,0.846296,0.846296,0.628312,17.833333,164.212,152.116,164.106196,64,0,1.166667,1.833333,2.500000,2.206795,2.082510,230.072993,8.225768,6.902119,6.902119,5.966326,4.193447,4.193447,2.785840,2.785840,1.951981,1.951981,1.319357,1.319357,-1.16
3,2-(1-piperazinyl)phenol,c1ccc(c(c1)N2CCNCC2)O,125.0,9.600688,9.600688,0.379074,0.379074,0.667456,17.384615,178.235,164.123,178.110613,70,0,1.153846,1.846154,2.538462,2.198429,2.183880,282.008164,9.096012,7.532255,7.532255,6.377010,4.614126,4.614126,3.225762,3.225762,2.339415,2.339415,1.637308,1.637308,-1.22
4,2-(1-cyclohexenyl)ethylamine,C1CCC(=CC1)CCN,-55.0,5.418520,5.418520,0.825231,0.825231,0.559754,19.444444,125.215,110.095,125.120449,52,0,1.444444,2.333333,2.888889,1.910086,2.281634,105.135263,6.527098,5.897341,5.897341,4.431852,3.812278,3.812278,2.646829,2.646829,1.850480,1.850480,1.317105,1.317105,-0.30


In [57]:
# Correlation matrix of the selected descriptors

corr_matrix = descriptor_data[top_descriptors].corr().abs()


upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)

)

threshold = 0.95

to_drop = [
    column
    for column in upper.columns
    if any(upper[column] > threshold)
]

print("Descriptors to remove:")
print(to_drop)

selected_descriptors = [
    desc for desc in top_descriptors
    if desc not in to_drop
]

print(f"Descriptors before : {len(top_descriptors)}")
print(f"Descriptors after  : {len(selected_descriptors)}")

data = pd.concat(
    [
        final_data[["name", "smiles", "mpC"]],
        descriptor_data[selected_descriptors]
    ],
    axis=1
)


Descriptors to remove:
['MaxEStateIndex', 'HeavyAtomMolWt', 'ExactMolWt', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n', 'Chi3n', 'Chi4n', 'Chi4v']
Descriptors before : 30
Descriptors after  : 17


In [58]:
data.head()

,name,smiles,mpC,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,NumValenceElectrons,NumRadicalElectrons,FpDensityMorgan1,FpDensityMorgan2,FpDensityMorgan3,AvgIpc,BalabanJ,BertzCT,Chi2v,Chi3v,HallKierAlpha
0,"2-(2,4-dinitrobenzyl)pyridine",c1ccnc(c1)Cc2ccc(cc2[N+](=O)[O-])[N+](=O)[O-],92.0,10.949337,0.259075,-0.655211,0.619888,10.105263,259.221,96,0,1.052632,1.842105,2.421053,2.394721,2.399145,628.126552,3.825087,2.574114,-2.83
1,2-(1-piperidinyl)aniline,c1ccc(c(c1)N)N2CCCCC2,46.0,5.909724,0.906852,0.906852,0.664933,17.384615,176.263,70,0,1.000000,1.692308,2.384615,2.198429,2.183880,277.173697,3.606524,2.633505,-1.18
2,2-(1-piperazinyl)pyrimidine,c1cnc(nc1)N2CCNCC2,33.0,4.189444,0.846296,0.846296,0.628312,17.833333,164.212,64,0,1.166667,1.833333,2.500000,2.206795,2.082510,230.072993,2.785840,1.951981,-1.16
3,2-(1-piperazinyl)phenol,c1ccc(c(c1)N2CCNCC2)O,125.0,9.600688,0.379074,0.379074,0.667456,17.384615,178.235,70,0,1.153846,1.846154,2.538462,2.198429,2.183880,282.008164,3.225762,2.339415,-1.22
4,2-(1-cyclohexenyl)ethylamine,C1CCC(=CC1)CCN,-55.0,5.418520,0.825231,0.825231,0.559754,19.444444,125.215,52,0,1.444444,2.333333,2.888889,1.910086,2.281634,105.135263,2.646829,1.850480,-0.30


From 217 total no of descriptors, choose top 30 of them on having a look at them any of them were actually coorrelated to each other we need to remove them as well so all those who where coorrelated we find top 17 desciptors

In [62]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
mpC,28344.0,94.396756,97.018968,-2.050000e+02,35.000000,97.000000,163.000000,804.000000
MaxAbsEStateIndex,28344.0,9.283005,3.272410,0.000000e+00,6.471745,10.410073,11.702279,17.808922
MinAbsEStateIndex,28344.0,0.405819,0.513606,0.000000e+00,0.087523,0.245938,0.565618,9.847222
MinEStateIndex,28344.0,-0.518435,1.439937,-1.124769e+01,-0.902990,-0.333333,0.370060,6.000000
qed,28344.0,0.567242,0.156861,1.979820e-02,0.463261,0.569267,0.677727,0.945893
SPS,28344.0,14.264463,9.482476,0.000000e+00,9.600000,10.666667,14.133333,108.000000
MolWt,28344.0,224.013884,106.986420,1.604300e+01,151.165000,200.295500,278.308000,1701.206000
NumValenceElectrons,28344.0,80.065340,39.487402,1.000000e+00,54.000000,70.000000,100.000000,632.000000
NumRadicalElectrons,28344.0,0.000529,0.027215,0.000000e+00,0.000000,0.000000,0.000000,3.000000
FpDensityMorgan1,28344.0,1.137714,0.346106,6.756757e-02,0.909091,1.157895,1.363636,4.000000
